In [ ]:
!pip install -q groq gradio

器材借用系統


In [ ]:
# library
import sqlite3
from google.colab import userdata #Important, misuse would trigger GitHub lock
from groq import Groq
import gradio as gr

In [ ]:
# API
client = Groq(api_key=userdata.get('SDDA_Week05'))
print('Successfully access Groq API')

Successfully access Groq API


In [ ]:
# database
conn = sqlite3.connect('equipment.db')
cursor = conn.cursor()
print("Successfully access SQLite database")

Successfully access SQLite database


In [ ]:
# database: prepare
cursor.execute('DROP TABLE IF EXISTS items')
cursor.execute('DROP TABLE IF EXISTS borrow_logs')

In [ ]:
# database: table building- items
cursor.execute("""
 CREATE TABLE items(
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT,
  status TEXT)
""")

In [ ]:
# database: table building- borrow_logs
cursor.execute("""
 CREATE TABLE borrow_logs(
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  item_name TEXT,
  borrow_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP)
""")

In [ ]:
# database: insert data for testing

initial_data = [('單眼相機','可借用'),('腳架','以借出'),('投影機','可借用')]
cursor.executemany('INSERT INTO items (name, status) VALUES (?, ?)', initial_data)
conn.commit()
conn.close()

print('Successfully INSERT items into table (items)')

Successfully INSERT items into table (items)


In [ ]:
def ai_find_items(user_say):
  prompt = f"""
  系辦目前有三種設備: [單眼相機、腳架、投影機]。
  請從學生的話裡面找出他想借用的哪種器材。
  規則: 只能回傳器材名稱(例如: 單眼相機、腳架、投影機)。如果學生說的不是這三種，請回傳[未知]。
  絕對不要給其他多餘的文字!

  學生的話: {user_say}
  """

  response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    temperature= 0.0
  )

  items_name = response.choices[0].message.content.strip()
  return items_name

In [ ]:
result = ai_find_items('系秘好，我下午要拍片，想要一台相機。')
print(result)

單眼相機


In [ ]:
user_say = '系秘好，我下午要拍片，想要一台相機。'

In [ ]:
# database: checking
def borrow_system(user_say):

# 1.check what student want to borrow by AI
  target = ai_find_items(user_say)
  print (f'try to find {target} in item... ')

  if target not in ['單眼相機','腳架','投影機']:
    return f'I can not find {target} in items database.'

  # 2. connect to db to check the item is available
  conn = sqlite3.connect('equipment.db')
  cursor = conn.cursor()

  cursor.execute('SELECT * FROM items WHERE name = ?', (target,))
  row = cursor.fetchone()

  current_stats = row[2]
  print (f'[RESULT] {target} status is {current_stats}!')

  # 3.check the item is available
  if current_stats == '以借出':
    conn.close()
    return f'Sorry, the {target} is not available.'
  else:
  # 4.執行借出程序，將"可借用"改成"以借出"
    cursor.execute("UPDATE items SET status = '以借出' WHERE name = ?", (target,))
    cursor.execute('INSERT INTO borrow_logs (item_name) VALUES (?)', (target,))

    conn.commit()
    conn.close()
    return f'Successfully borrow {target}, and change the ststus into 以借出'

In [ ]:
# @title
print (borrow_system('我想借單眼相機'))

try to find 單眼相機 in item... 
[RESULT] 單眼相機 status is 以借出!
Sorry, the 單眼相機 is not available.


In [ ]:
def web_interface(message):
    if not message.strip():
        return "請輸入想借用什麼器材！"
    reply = borrow_system(message)
    return reply


demo = gr.Interface(
    fn=web_interface,
    inputs=gr.Textbox(label="輸入您的需求"),
    outputs=gr.Textbox(label="系統回應"),
    title ='簡易器材借用系統',
    description= '結合Groq與SQLite的器材借用處'
)

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://935021313235b43f70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


try to find 投影機 in item... 
[RESULT] 投影機 status is 可借用!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://935021313235b43f70.gradio.live
